# 16 - LSTM with validation choice, and cross-system comparison

1. Choose the LSTM's k on a validation part.
2. LSTM on the unlabelled OpenSSH and Apache windows, compared with the Isolation Forest flags.
3. One table with every model on every dataset, and what happens when a setting chosen on one dataset is used on the other.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.cross_system import chosen_rows, chosen_settings, transfer_table
from src.evaluation.metrics import evaluate
from src.models.lstm import build_vocab, evenly_spaced, fit_lstm, flag, known_events, miss_scores, sweep_k
from src.models.supervised import select_rf_method
from src.models.unsupervised import (IForestDetector, PCADetector, score_unlabelled, time_split, time_split_val,
                                     top_flagged, validated_sweeps)

pd.set_option("display.max_colwidth", 200)
DOCS, PROCESSED, MODELS = ROOT / "docs", ROOT / "data" / "processed", ROOT / "models"
H, SEED, N_TRAIN = 10, 42, 20000
MODEL_CONFIG = {"embed_dim": 32, "hidden_size": 64, "num_layers": 2}
KS = [1, 2, 3, 5, 9, 12]
PERCENTILES = [90, 95, 97.5, 99, 99.5]
LABELLED = {"HDFS": ("hdfs", "order"), "BGL": ("bgl", "window")}
UNLABELLED = {"OpenSSH": "openssh", "Apache": "apache"}


def templates_of(prefix):
    return pd.read_csv(PROCESSED / f"{prefix}_templates.csv").set_index("EventId")["template"]

## LSTM: choose k on validation

Same model and training as notebook 12 (h = 10, 2 layers, hidden size 64, early stopping on 10% of the training sequences), trained on 20,000 evenly spaced normal sequences from the first 60%. Events never seen in these sequences are flagged (m = 1). k is chosen by F1 on the next 10%.

In [2]:
lstm, lstm_val, lstm_test = {}, [], []
for name, (prefix, order_col) in LABELLED.items():
    sequences = pd.read_parquet(PROCESSED / f"{prefix}_sequences.parquet")
    fit, val, test = time_split_val(sequences, order_col=order_col)
    train_seqs = evenly_spaced(fit.loc[fit["Label"] == 0, "EventSequence"].tolist(), N_TRAIN)
    vocab = build_vocab(templates_of(prefix).index)
    model, history, seconds = fit_lstm(train_seqs, vocab, H, SEED, model_config=MODEL_CONFIG)
    known = known_events(train_seqs, 1)
    print(f"{name}: {len(train_seqs):,} training sequences, {len(history)} epochs, {seconds:.0f} s, "
          f"{len(known)} of {len(vocab)} event types seen")

    v, *_ = sweep_k(model, val["EventSequence"].tolist(), vocab, known, KS, val["Label"].to_numpy(), H)
    t, _, ranks, seq_idx = sweep_k(model, test["EventSequence"].tolist(), vocab, known, KS, test["Label"].to_numpy(), H)
    lstm_val.append(v.assign(dataset=name, model="LSTM"))
    lstm_test.append(t.assign(dataset=name, model="LSTM"))
    lstm[name] = {"train": train_seqs, "test": test, "ranks": ranks, "seq_idx": seq_idx}
    MODELS.mkdir(exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "vocab": vocab, "train_events": sorted(known),
                "config": {"h": H, **MODEL_CONFIG}}, MODELS / f"{prefix}_lstm_fit60.pt")

lstm_val, lstm_test = pd.concat(lstm_val, ignore_index=True), pd.concat(lstm_test, ignore_index=True)
lstm_val.pivot_table(index="dataset", columns="k", values="f1", sort=False).round(3)

epoch  1  train loss 0.2965  val loss 0.2918


epoch  2  train loss 0.2646  val loss 0.2649


epoch  3  train loss 0.2506  val loss 0.2528


epoch  4  train loss 0.2468  val loss 0.2502


epoch  5  train loss 0.2466  val loss 0.2499


epoch  6  train loss 0.2442  val loss 0.2478


... (25 outputs left out)

epoch 13  train loss 0.0633  val loss 0.0831


epoch 14  train loss 0.0631  val loss 0.0837


epoch 15  train loss 0.0627  val loss 0.0831
early stopping after epoch 15
BGL: 20,000 training sequences, 15 epochs, 253 s, 130 of 391 event types seen


k,1,2,3,5,9,12
dataset,,,,,,
HDFS,0.090,0.321,0.381,0.532,0.463,0.430
BGL,0.241,0.256,0.268,0.276,0.286,0.289


Validation F1 by k (above). Test results for the chosen k:

In [3]:
lstm_chosen = chosen_rows(lstm_val, lstm_test, "k")
lstm_chosen[["dataset", "k", "val_f1", "precision", "recall", "f1", "false_alarm_rate", "flag_rate"]].style.format({c: "{:.4f}" for c in ["precision", "recall", "f1", "false_alarm_rate", "val_f1"]} | {"flag_rate": "{:.2%}"}).hide(axis="index")

dataset,k,val_f1,precision,recall,f1,false_alarm_rate,flag_rate
HDFS,5,0.5321,0.8017,0.4300,0.5598,0.0023,1.16%
BGL,12,0.2890,0.0974,0.9925,0.1774,0.7410,75.97%


## Rare-event rule with m = 13 (HDFS)

Events seen fewer than 13 times in the training sequences also count as unseen. Same model and chosen k.

In [4]:
h, k = lstm["HDFS"], chosen_settings(lstm_val, "k")[("HDFS", "LSTM")]
test_seqs, y_test = h["test"]["EventSequence"].tolist(), h["test"]["Label"].to_numpy()
rows, preds = [], {}
for m in [1, 13]:
    known_m = known_events(h["train"], m)
    preds[m] = flag(test_seqs, h["ranks"], h["seq_idx"], known_m, k)
    r = evaluate(y_test, preds[m])
    rows.append({"m": m, "known events": len(known_m), "precision": r["precision"], "recall": r["recall"], "f1": r["f1"],
                 "false_alarm_rate": r["false_alarm_rate"]})
rare = sorted(known_events(h["train"], 1) - known_events(h["train"], 13), key=lambda e: int(e[1:]))
print(f"k = {k}. Events seen 1-12 times, unseen with m = 13: {rare or 'none'}")
print(f"Test blocks whose flag changes: {(preds[1] != preds[13]).sum():,}")
pd.DataFrame(rows).style.format({c: "{:.4f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]}).hide(axis="index")

k = 5. Events seen 1-12 times, unseen with m = 13: ['E8', 'E9', 'E34']
Test blocks whose flag changes: 0


m,known events,precision,recall,f1,false_alarm_rate
1,18,0.8017,0.4300,0.5598,0.0023
13,15,0.8017,0.4300,0.5598,0.0023


## LSTM on OpenSSH and Apache

No labels. Trained on all windows of the first 70% (fewer than 20,000), scored on the last 30%. To rank flagged windows, a window's score is the share of its events that are unseen or outside the top-9 predictions.

In [5]:
unlabelled = {}
for name, prefix in UNLABELLED.items():
    sequences = pd.read_parquet(PROCESSED / f"{prefix}_sequences.parquet")
    counts = pd.read_parquet(PROCESSED / f"{prefix}_counts.parquet")
    templates = templates_of(prefix)
    train, test = time_split(sequences, train_frac=0.7, order_col="window")
    train_seqs, test_seqs = train["EventSequence"].tolist(), test["EventSequence"].tolist()
    vocab = build_vocab(templates.index)
    model, history, seconds = fit_lstm(train_seqs, vocab, H, SEED, model_config=MODEL_CONFIG)
    known = known_events(train_seqs, 1)
    print(f"{name}: {len(train_seqs):,} training windows, {len(history)} epochs, {seconds:.0f} s, "
          f"{len(known)} of {len(vocab)} event types seen")

    rates, _, ranks, seq_idx = sweep_k(model, test_seqs, vocab, known, KS, None, H)
    if_test, if_scores, if_train_scores = score_unlabelled(counts, IForestDetector(max_samples=1.0, flag_unseen=True))
    assert (if_test["window"].to_numpy() == test["window"].to_numpy()).all()
    unlabelled[name] = {"test": test.reset_index(drop=True), "templates": templates, "known": known, "rates": rates,
                        "lstm_scores": miss_scores(test_seqs, ranks, seq_idx, known, 9),
                        "if_scores": if_scores, "if_train_scores": if_train_scores, "counts": counts}

pd.DataFrame({name: u["rates"].set_index("k").loc[[5, 9], "flag_rate"] for name, u in unlabelled.items()}).T \
    .rename(columns=lambda k: f"flagged at k = {k}").style.format("{:.1%}")

epoch  1  train loss 0.3572  val loss 0.3123


epoch  2  train loss 0.3007  val loss 0.2636


epoch  3  train loss 0.2795  val loss 0.2459


epoch  4  train loss 0.2637  val loss 0.2350


epoch  5  train loss 0.2580  val loss 0.2311


epoch  6  train loss 0.2497  val loss 0.2258


... (46 outputs left out)

epoch 28  train loss 0.5974  val loss 0.6184


epoch 29  train loss 0.5958  val loss 0.6209


epoch 30  train loss 0.5923  val loss 0.6175
Apache: 1,976 training windows, 30 epochs, 48 s, 29 of 30 event types seen


k,flagged at k = 5,flagged at k = 9
OpenSSH,20.4%,9.0%
Apache,19.6%,3.5%


Top 5 flagged OpenSSH windows (LSTM):

In [6]:
u = unlabelled["OpenSSH"]
top_flagged(u["test"], u["lstm_scores"], u["templates"], known_events=u["known"]).style.format({"score": "{:.3f}"}, escape="html").hide(axis="index")

window,score,lines,top events,unseen events
12933,1.000,2,E49 x1: Received disconnect from <IP>: <NUM>: Disconnect requested by Windows SSH Client.; E23 x1: pam_unix(sshd:session): session closed for user <USER>,E49: Received disconnect from <IP>: <NUM>: Disconnect requested by Windows SSH Client.
12906,0.500,2,E32 x2: Bad protocol version identification <*> from <IP> port <PORT>,
12929,0.500,2,E15 x1: Did not receive identification string from <IP>; E23 x1: pam_unix(sshd:session): session closed for user <USER>,
13090,0.500,2,E23 x1: pam_unix(sshd:session): session closed for user <USER>; E51 x1: syslogin_perform_logout: logout() returned an error,E51: syslogin_perform_logout: logout() returned an error
13252,0.500,2,E10 x1: Failed password for <USER> from <IP> port <PORT> ssh2; E25 x1: fatal: Read from socket failed: Connection reset by peer [preauth],


Top 5 flagged Apache windows (LSTM):

In [7]:
u = unlabelled["Apache"]
top_flagged(u["test"], u["lstm_scores"], u["templates"], known_events=u["known"]).style.format({"score": "{:.3f}"}, escape="html").hide(axis="index")

window,score,lines,top events,unseen events
2198,0.450,20,E30 x8: [client <IP>] client sent HTTP/<NUM>.<NUM> request without hostname (see RFC2616 section <NUM>.<NUM>): /; E14 x6: [client <IP>] Directory index forbidden by rule: /var/www/html/; E15 x4: [client <IP>] File does not exist: <*>,E30: [client <IP>] client sent HTTP/<NUM>.<NUM> request without hostname (see RFC2616 section <NUM>.<NUM>): /
2612,0.100,20,E14 x15: [client <IP>] Directory index forbidden by rule: /var/www/html/; E24 x5: script not found or unable to stat,
2025,0.050,20,"E15 x18: [client <IP>] File does not exist: <*>; E20 x1: Graceful restart requested, doing restart; E19 x1: mod_jk2 Shutting down",
2087,0.050,20,E14 x9: [client <IP>] Directory index forbidden by rule: /var/www/html/; E15 x8: [client <IP>] File does not exist: <*>; E24 x3: script not found or unable to stat,
2126,0.050,20,E12 x6: workerEnv.init() ok /etc/httpd/conf/workers2.properties; E13 x5: mod_jk child init <NUM> <NUM>; E16 x5: jk2_init() Can't find child <NUM> in scoreboard,


### Agreement with Isolation Forest

Isolation Forest as in notebook 14 (`max_samples=1.0`, `flag_unseen=True`). Overlap of the highest-ranked windows, and of the flagged sets (LSTM k = 9, Isolation Forest 99th percentile).

In [8]:
rows = []
for name, u in unlabelled.items():
    windows = u["test"]["window"].to_numpy()
    top = lambda scores, n: set(windows[np.argsort(-scores, kind="stable")[:n]])
    lstm_flagged = u["lstm_scores"] > 0  # any unseen event or miss at k = 9, the same as the k = 9 flags
    if_flagged = u["if_scores"] > np.percentile(u["if_train_scores"], 99)
    rows.append({"dataset": name,
                 "top 5 in both": len(top(u["lstm_scores"], 5) & top(u["if_scores"], 5)),
                 "top 50 in both": len(top(u["lstm_scores"], 50) & top(u["if_scores"], 50)),
                 "LSTM flags (k = 9)": int(lstm_flagged.sum()), "IF flags (p99)": int(if_flagged.sum()),
                 "flagged by both": int((lstm_flagged & if_flagged).sum())})
pd.DataFrame(rows).style.hide(axis="index")

dataset,top 5 in both,top 50 in both,LSTM flags (k = 9),IF flags (p99),flagged by both
OpenSSH,0,35,363,122,79
Apache,1,4,30,4,1


## Cross-system table

Every model on every dataset with its validation-chosen setting. Isolation Forest is the fixed version from notebook 14; Random Forest chooses between 'none', 'balanced' and 'smote' (notebook 15). OpenSSH and Apache have no labels, so nothing can be chosen on them: they get the flag rate with the setting chosen on HDFS and with the one chosen on BGL. Random Forest needs labels, so it is not run on them.

In [9]:
detectors = lambda: {"PCA": PCADetector(variance=0.95), "Isolation Forest": IForestDetector(max_samples=1.0, flag_unseen=True)}
unsup_val, unsup_test = [], []
for name, (prefix, order_col) in LABELLED.items():
    v, t = validated_sweeps(pd.read_parquet(PROCESSED / f"{prefix}_counts.parquet"), detectors(), PERCENTILES, order_col)
    unsup_val.append(v.assign(dataset=name))
    unsup_test.append(t.assign(dataset=name))
unsup_val, unsup_test = pd.concat(unsup_val, ignore_index=True), pd.concat(unsup_test, ignore_index=True)

rows = []
for r in chosen_rows(unsup_val, unsup_test, "percentile").itertuples():
    rows.append({"dataset": r.dataset, "model": r.model, "setting": f"percentile {r.percentile:g}", "chosen on": r.dataset,
                 "precision": r.precision, "recall": r.recall, "f1": r.f1, "false_alarm_rate": r.false_alarm_rate,
                 "flag_rate": r.flag_rate})
for name, (prefix, order_col) in LABELLED.items():
    rf = select_rf_method(pd.read_parquet(PROCESSED / f"{prefix}_counts.parquet"), order_col=order_col)["test"]
    rows.append({"dataset": name, "model": "Random Forest", "setting": f"method {rf['method']}", "chosen on": name,
                 "precision": rf["precision"], "recall": rf["recall"], "f1": rf["f1"], "false_alarm_rate": rf["false_alarm_rate"],
                 "flag_rate": (rf["tp"] + rf["fp"]) / (rf["tp"] + rf["fp"] + rf["tn"] + rf["fn"])})
for r in lstm_chosen.itertuples():
    rows.append({"dataset": r.dataset, "model": "LSTM", "setting": f"k {r.k}", "chosen on": r.dataset,
                 "precision": r.precision, "recall": r.recall, "f1": r.f1, "false_alarm_rate": r.false_alarm_rate,
                 "flag_rate": r.flag_rate})

percentiles, ks = chosen_settings(unsup_val, "percentile"), chosen_settings(lstm_val, "k")
for name, u in unlabelled.items():
    pca_test, pca_scores, pca_train_scores = score_unlabelled(u["counts"], PCADetector(variance=0.95))
    scores = {"PCA": (pca_scores, pca_train_scores), "Isolation Forest": (u["if_scores"], u["if_train_scores"])}
    for source in LABELLED:
        for model, (test_scores, train_scores) in scores.items():
            p = percentiles[(source, model)]
            rows.append({"dataset": name, "model": model, "setting": f"percentile {p:g}", "chosen on": source,
                         "flag_rate": (test_scores > np.percentile(train_scores, p)).mean()})
        k = ks[(source, "LSTM")]
        rows.append({"dataset": name, "model": "LSTM", "setting": f"k {k}", "chosen on": source,
                     "flag_rate": u["rates"].set_index("k").loc[k, "flag_rate"]})

order = {"PCA": 0, "Isolation Forest": 1, "Random Forest": 2, "LSTM": 3}
cross = pd.DataFrame(rows)
cross = cross.sort_values(["dataset", "model"], key=lambda c: c.map({"HDFS": 0, "BGL": 1, "OpenSSH": 2, "Apache": 3} | order),
                          kind="stable").reset_index(drop=True)
cross.round(4).to_csv(DOCS / "cross_system_results.csv", index=False)
cross.style.format({c: "{:.4f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]} | {"flag_rate": "{:.2%}"},
                   na_rep="-").hide(axis="index")

dataset,model,setting,chosen on,precision,recall,f1,false_alarm_rate,flag_rate
HDFS,PCA,percentile 95,HDFS,0.8598,0.8174,0.8381,0.0029,2.05%
HDFS,Isolation Forest,percentile 99,HDFS,0.7873,0.5111,0.6199,0.0030,1.40%
HDFS,Random Forest,method balanced,HDFS,1.0000,0.9970,0.9985,0.0000,2.15%
HDFS,LSTM,k 5,HDFS,0.8017,0.4300,0.5598,0.0023,1.16%
BGL,PCA,percentile 99,BGL,0.0947,0.9799,0.1727,0.7548,77.16%
BGL,Isolation Forest,percentile 99.5,BGL,0.0960,0.9908,0.1750,0.7522,77.00%
BGL,Random Forest,method balanced,BGL,0.9551,0.0721,0.1341,0.0003,0.56%
BGL,LSTM,k 12,BGL,0.0974,0.9925,0.1774,0.7410,75.97%
OpenSSH,PCA,percentile 95,HDFS,-,-,-,-,3.81%
OpenSSH,PCA,percentile 99,BGL,-,-,-,-,1.41%


## Transfer: settings chosen on the other dataset

Each model is still fitted on its own dataset; only the threshold percentile (PCA, Isolation Forest) or k (LSTM) comes from the other dataset's validation choice.

In [10]:
transfer = pd.concat([transfer_table(unsup_val, unsup_test, "percentile"), transfer_table(lstm_val, lstm_test, "k")],
                     ignore_index=True).sort_values(["applied to", "model"], key=lambda c: c.map({"HDFS": 0, "BGL": 1} | order), kind="stable")
transfer.round(4).to_csv(DOCS / "cross_system_transfer.csv", index=False)
transfer.style.format({c: "{:.4f}" for c in ["own f1", "transferred f1", "transferred precision", "transferred recall",
                                             "transferred false_alarm_rate"]} | {"f1 change": "{:+.4f}"}).hide(axis="index")

model,applied to,setting from,own setting,own f1,transferred setting,transferred f1,f1 change,transferred precision,transferred recall,transferred false_alarm_rate
PCA,HDFS,BGL,95.000000,0.8381,99.000000,0.8381,+0.0000,0.8598,0.8174,0.0029
Isolation Forest,HDFS,BGL,99.000000,0.6199,99.500000,0.6199,+0.0000,0.7873,0.5111,0.0030
LSTM,HDFS,BGL,5.000000,0.5598,12.000000,0.5465,-0.0133,0.9758,0.3795,0.0002
PCA,BGL,HDFS,99.000000,0.1727,95.000000,0.1661,-0.0066,0.0906,0.9991,0.8084
Isolation Forest,BGL,HDFS,99.500000,0.1750,99.000000,0.1749,-0.0001,0.0959,0.9910,0.7530
LSTM,BGL,HDFS,12.000000,0.1774,5.000000,0.1730,-0.0045,0.0947,0.9925,0.7644
